In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AffinityPropagation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import sys, os

project_root = 'c:/big20/git/big20-ML-project2-team3/OpionReview'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from utils.preprocessing import get_default_data



# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)


class ImprovedAPAnalyzer:
    """
    개선된 Affinity Propagation 분석기
    - 군집 수 제어
    - 다양한 유사도 메트릭
    - 기존 전처리 함수 활용
    """
    
    def __init__(self):
        self.document_df = None
        self.tfidf_matrix = None
        self.documents = None
        self.labels = None
        self.exemplars = None
        self.ap_model = None
        self.similarity_matrix = None
        
    def load_data(self):
        """
        기존 전처리 함수로 데이터 로드
        """
        print("="*80)
        print("📂 데이터 로딩 중...")
        print("="*80)
        
        # 기존 전처리 함수 사용
        self.document_df = preprocessing.get_default_data()
        
        print(f"✅ 데이터 로드 완료")
        print(f"   - 문서 수: {len(self.document_df)}")
        print(f"   - 컬럼: {list(self.document_df.columns)}")
        
        # TF-IDF 행렬 추출 (전처리 함수에서 생성된 것 사용)
        if 'tfidf_matrix' in dir(preprocessing):
            self.tfidf_matrix = preprocessing.tfidf_matrix
        elif hasattr(self.document_df, 'tfidf_matrix'):
            self.tfidf_matrix = self.document_df.tfidf_matrix
        else:
            print("⚠️  TF-IDF 행렬을 찾을 수 없습니다. 수동으로 생성합니다.")
            self._create_tfidf_matrix()
        
        print(f"   - TF-IDF Matrix 크기: {self.tfidf_matrix.shape}")
        print("="*80 + "\n")
        
        return self.document_df
    
    def _create_tfidf_matrix(self):
        """
        TF-IDF 행렬 수동 생성 (필요시)
        """
        from sklearn.feature_extraction.text import TfidfVectorizer
        
        # 문서 컬럼 찾기
        text_column = None
        for col in ['document', 'text', 'review', 'content']:
            if col in self.document_df.columns:
                text_column = col
                break
        
        if text_column is None:
            raise ValueError("문서 텍스트 컬럼을 찾을 수 없습니다.")
        
        self.documents = self.document_df[text_column].tolist()
        
        vectorizer = TfidfVectorizer(
            max_features=500,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.8,
            stop_words='english'
        )
        
        self.tfidf_matrix = vectorizer.fit_transform(self.documents)
    
    def calculate_similarity_matrix(self, method='cosine'):
        """
        다양한 방법으로 유사도 행렬 계산
        
        Parameters:
        -----------
        method : str
            'cosine': 코사인 유사도 (기본, 0~1)
            'euclidean': 유클리드 거리 기반 (-무한대~0, 가까울수록 0)
            'manhattan': 맨하탄 거리 기반
            'pearson': 피어슨 상관계수 (-1~1)
            'rbf': RBF 커널 (0~1)
        """
        print(f"🔧 유사도 행렬 계산 중 (method={method})...")
        
        if method == 'cosine':
            # 코사인 유사도 (높을수록 유사)
            self.similarity_matrix = cosine_similarity(self.tfidf_matrix)
            
        elif method == 'euclidean':
            # 유클리드 거리 (낮을수록 유사) -> 음수로 변환
            distances = euclidean_distances(self.tfidf_matrix)
            self.similarity_matrix = -distances
            
        elif method == 'manhattan':
            # 맨하탄 거리 (낮을수록 유사) -> 음수로 변환
            distances = manhattan_distances(self.tfidf_matrix)
            self.similarity_matrix = -distances
            
        elif method == 'pearson':
            # 피어슨 상관계수
            from scipy.stats import pearsonr
            dense_matrix = self.tfidf_matrix.toarray()
            n = dense_matrix.shape[0]
            self.similarity_matrix = np.zeros((n, n))
            
            for i in range(n):
                for j in range(i, n):
                    if i == j:
                        self.similarity_matrix[i, j] = 1.0
                    else:
                        corr, _ = pearsonr(dense_matrix[i], dense_matrix[j])
                        self.similarity_matrix[i, j] = corr
                        self.similarity_matrix[j, i] = corr
                        
        elif method == 'rbf':
            # RBF (Radial Basis Function) 커널
            from sklearn.metrics.pairwise import rbf_kernel
            # gamma 값 조정으로 민감도 조절 (작을수록 덜 민감)
            gamma = 0.1  # 기본값보다 작게 설정
            self.similarity_matrix = rbf_kernel(self.tfidf_matrix, gamma=gamma)
            
        else:
            raise ValueError(f"지원하지 않는 방법: {method}")
        
        print(f"✅ 유사도 행렬 계산 완료")
        print(f"   - 크기: {self.similarity_matrix.shape}")
        print(f"   - 최소값: {self.similarity_matrix.min():.4f}")
        print(f"   - 최대값: {self.similarity_matrix.max():.4f}")
        print(f"   - 평균값: {self.similarity_matrix.mean():.4f}")
        print(f"   - 중앙값: {np.median(self.similarity_matrix):.4f}\n")
        
        return self.similarity_matrix
    
    def find_optimal_preference(self, target_clusters=20, method='cosine',
                               min_clusters=10, max_clusters=30):
        """
        목표 군집 수에 맞는 최적 preference 값 탐색
        
        Parameters:
        -----------
        target_clusters : int
            목표 군집 수
        method : str
            유사도 계산 방법
        min_clusters : int
            최소 군집 수
        max_clusters : int
            최대 군집 수
        """
        print("="*80)
        print(f"🎯 최적 Preference 탐색 중 (목표 군집 수: {target_clusters})")
        print("="*80 + "\n")
        
        # 유사도 행렬 계산
        if self.similarity_matrix is None:
            self.calculate_similarity_matrix(method=method)
        
        # Preference 후보 값들
        percentiles = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
        preferences = [np.percentile(self.similarity_matrix, p) for p in percentiles]
        
        results = []
        
        print("Preference 값별 군집 수 테스트:")
        print("-" * 60)
        
        for pref in preferences:
            try:
                ap = AffinityPropagation(
                    damping=0.9,
                    max_iter=500,
                    preference=pref,
                    affinity='precomputed',
                    random_state=42,
                    verbose=False
                )
                
                labels = ap.fit_predict(self.similarity_matrix)
                n_clusters = len(set(labels))
                
                results.append({
                    'preference': pref,
                    'n_clusters': n_clusters,
                    'diff': abs(n_clusters - target_clusters)
                })
                
                status = "✅" if min_clusters <= n_clusters <= max_clusters else "⚠️"
                print(f"{status} Preference: {pref:8.4f} → 군집 수: {n_clusters:3d}")
                
            except Exception as e:
                print(f"❌ Preference: {pref:8.4f} → 실패: {e}")
        
        # 최적값 선택
        results_df = pd.DataFrame(results)
        best_result = results_df.loc[results_df['diff'].idxmin()]
        
        print("\n" + "="*60)
        print(f"✅ 최적 Preference: {best_result['preference']:.4f}")
        print(f"   - 군집 수: {best_result['n_clusters']}")
        print("="*60 + "\n")
        
        return best_result['preference'], results_df
    
    def fit_affinity_propagation(self, preference=None, damping=0.9, 
                                 max_iter=500, method='cosine',
                                 target_clusters=20):
        """
        Affinity Propagation 클러스터링
        
        Parameters:
        -----------
        preference : float or None
            Preference 값 (None이면 자동 탐색)
        damping : float
            감쇠 계수 (0.5~1.0)
        max_iter : int
            최대 반복 횟수
        method : str
            유사도 계산 방법
        target_clusters : int
            목표 군집 수 (preference=None일 때)
        """
        print("="*80)
        print("🎯 Affinity Propagation 클러스터링")
        print("="*80 + "\n")
        
        # 유사도 행렬 계산
        self.calculate_similarity_matrix(method=method)
        
        # Preference 자동 탐색
        if preference is None:
            print("⚙️  Preference 값을 자동으로 탐색합니다...\n")
            preference, search_results = self.find_optimal_preference(
                target_clusters=target_clusters,
                method=method
            )
        
        print(f"📊 AP 모델 설정:")
        print(f"   - Damping: {damping}")
        print(f"   - Max Iterations: {max_iter}")
        print(f"   - Preference: {preference:.4f}")
        print(f"   - Similarity Method: {method}\n")
        
        # AP 모델 학습
        self.ap_model = AffinityPropagation(
            damping=damping,
            max_iter=max_iter,
            preference=preference,
            affinity='precomputed',
            random_state=42,
            verbose=False
        )
        
        self.labels = self.ap_model.fit_predict(self.similarity_matrix)
        self.exemplars = self.ap_model.cluster_centers_indices_
        
        n_clusters = len(self.exemplars)
        
        print(f"✅ 클러스터링 완료")
        print(f"   - 발견된 군집 수: {n_clusters}")
        print(f"   - Exemplar 문서 수: {len(self.exemplars)}")
        print(f"   - 반복 횟수: {self.ap_model.n_iter_}")
        print("="*80 + "\n")
        
        # 군집 분포 출력
        self._print_cluster_distribution()
        
        return self.labels
    
    def _print_cluster_distribution(self):
        """
        군집 분포 출력
        """
        cluster_counts = Counter(self.labels)
        
        print("📊 군집 분포:")
        print("-" * 60)
        
        # 크기 순으로 정렬
        sorted_clusters = sorted(cluster_counts.items(), key=lambda x: x[1], reverse=True)
        
        for cluster_id, count in sorted_clusters[:20]:  # 상위 20개만
            percentage = (count / len(self.labels)) * 100
            bar = '█' * int(percentage / 2)
            print(f"  군집 {cluster_id:3d}: {count:4d}개 ({percentage:5.1f}%) {bar}")
        
        if len(sorted_clusters) > 20:
            remaining = sum(count for _, count in sorted_clusters[20:])
            print(f"  ... 외 {len(sorted_clusters)-20}개 군집: {remaining}개")
        
        print()
    
    def compare_similarity_methods(self, methods=['cosine', 'euclidean', 'manhattan', 'rbf'],
                                   target_clusters=20):
        """
        여러 유사도 방법 비교
        
        Parameters:
        -----------
        methods : list
            비교할 유사도 방법들
        target_clusters : int
            목표 군집 수
        """
        print("="*80)
        print("🔬 유사도 방법 비교 실험")
        print("="*80 + "\n")
        
        comparison_results = []
        
        for method in methods:
            print(f"\n{'='*60}")
            print(f"방법: {method.upper()}")
            print(f"{'='*60}\n")
            
            try:
                # 유사도 행렬 계산
                self.calculate_similarity_matrix(method=method)
                
                # 최적 preference 탐색
                opt_pref, search_df = self.find_optimal_preference(
                    target_clusters=target_clusters,
                    method=method
                )
                
                # 클러스터링
                labels = self.fit_affinity_propagation(
                    preference=opt_pref,
                    method=method,
                    target_clusters=target_clusters
                )
                
                n_clusters = len(set(labels))
                
                # 군집 품질 평가
                from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
                
                if n_clusters > 1:
                    silhouette = silhouette_score(self.tfidf_matrix.toarray(), labels)
                    calinski = calinski_harabasz_score(self.tfidf_matrix.toarray(), labels)
                    davies_bouldin = davies_bouldin_score(self.tfidf_matrix.toarray(), labels)
                else:
                    silhouette = 0.0
                    calinski = 0.0
                    davies_bouldin = 0.0
                
                comparison_results.append({
                    'Method': method,
                    'N_Clusters': n_clusters,
                    'Preference': opt_pref,
                    'Silhouette': silhouette,
                    'Calinski_Harabasz': calinski,
                    'Davies_Bouldin': davies_bouldin
                })
                
                print(f"\n✅ {method} 결과:")
                print(f"   - 군집 수: {n_clusters}")
                print(f"   - Silhouette: {silhouette:.4f}")
                print(f"   - Calinski-Harabasz: {calinski:.4f}")
                print(f"   - Davies-Bouldin: {davies_bouldin:.4f}")
                
            except Exception as e:
                print(f"\n❌ {method} 실패: {e}")
        
        # 결과 정리
        print("\n" + "="*80)
        print("📊 유사도 방법 비교 결과")
        print("="*80 + "\n")
        
        comparison_df = pd.DataFrame(comparison_results)
        print(comparison_df.to_string(index=False))
        print()
        
        # 추천
        if not comparison_df.empty:
            # Silhouette Score가 가장 높은 방법 추천
            best_method = comparison_df.loc[comparison_df['Silhouette'].idxmax()]
            print(f"🏆 추천 방법: {best_method['Method'].upper()}")
            print(f"   - 군집 수: {int(best_method['N_Clusters'])}")
            print(f"   - Silhouette Score: {best_method['Silhouette']:.4f}")
        
        print("="*80 + "\n")
        
        return comparison_df
    
    def visualize_clusters(self, method='tsne', figsize=(14, 10), save_path=None):
        """
        군집 시각화
        
        Parameters:
        -----------
        method : str
            'tsne' or 'pca'
        """
        if self.labels is None:
            raise ValueError("먼저 fit_affinity_propagation()을 실행하세요.")
        
        print(f"🎨 {method.upper()} 차원 축소 중...")
        
        # 차원 축소
        if method == 'tsne':
            reducer = TSNE(n_components=2, random_state=42, 
                          perplexity=min(30, len(self.document_df)-1))
        else:
            reducer = PCA(n_components=2, random_state=42)
        
        coords = reducer.fit_transform(self.tfidf_matrix.toarray())
        
        # 시각화
        fig, ax = plt.subplots(figsize=figsize)
        
        n_clusters = len(set(self.labels))
        colors = plt.cm.tab20(np.linspace(0, 1, min(n_clusters, 20)))
        
        # 큰 군집부터 그리기
        cluster_sizes = Counter(self.labels)
        sorted_clusters = [c for c, _ in cluster_sizes.most_common()]
        
        for i, cluster_id in enumerate(sorted_clusters[:20]):  # 상위 20개만
            mask = self.labels == cluster_id
            cluster_coords = coords[mask]
            
            color_idx = i % 20
            ax.scatter(cluster_coords[:, 0], cluster_coords[:, 1],
                      c=[colors[color_idx]], label=f'군집 {cluster_id} ({sum(mask)}개)',
                      alpha=0.6, s=80)
            
            # Exemplar 표시
            if cluster_id < len(self.exemplars):
                exemplar_idx = self.exemplars[cluster_id]
                exemplar_coord = coords[exemplar_idx]
                ax.scatter(exemplar_coord[0], exemplar_coord[1],
                          c=[colors[color_idx]], marker='*', s=500,
                          edgecolors='black', linewidths=2, zorder=10)
        
        # 나머지 군집들은 회색으로
        if n_clusters > 20:
            remaining_mask = ~np.isin(self.labels, sorted_clusters[:20])
            remaining_coords = coords[remaining_mask]
            ax.scatter(remaining_coords[:, 0], remaining_coords[:, 1],
                      c='lightgray', label=f'기타 ({sum(remaining_mask)}개)',
                      alpha=0.3, s=50)
        
        ax.set_title(f'AP 클러스터링 결과 ({method.upper()}) - 총 {n_clusters}개 군집', 
                    fontsize=16, fontweight='bold')
        ax.set_xlabel(f'{method.upper()} 1', fontsize=12)
        ax.set_ylabel(f'{method.upper()} 2', fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"💾 그래프 저장: {save_path}")
        
        plt.show()
    
    def get_cluster_summary(self, cluster_id, top_n_keywords=10):
        """
        특정 군집 요약 정보
        """
        if self.labels is None:
            raise ValueError("먼저 fit_affinity_propagation()을 실행하세요.")
        
        cluster_docs = np.where(self.labels == cluster_id)[0]
        
        # Exemplar
        exemplar_idx = self.exemplars[cluster_id] if cluster_id < len(self.exemplars) else cluster_docs[0]
        
        # 키워드 추출 (TF-IDF 평균)
        cluster_tfidf = self.tfidf_matrix[cluster_docs].mean(axis=0).A1
        
        # 상위 키워드
        if hasattr(preprocessing, 'vectorizer'):
            feature_names = preprocessing.vectorizer.get_feature_names_out()
            top_indices = cluster_tfidf.argsort()[-top_n_keywords:][::-1]
            top_keywords = [(feature_names[i], cluster_tfidf[i]) for i in top_indices]
        else:
            top_keywords = []
        
        summary = {
            'cluster_id': cluster_id,
            'n_documents': len(cluster_docs),
            'exemplar_idx': exemplar_idx,
            'top_keywords': top_keywords,
            'document_indices': cluster_docs.tolist()
        }
        
        return summary
    
    def export_results(self, output_path='ap_clustering_results.csv'):
        """
        결과 내보내기
        """
        if self.labels is None:
            raise ValueError("먼저 fit_affinity_propagation()을 실행하세요.")
        
        result_df = self.document_df.copy()
        result_df['cluster'] = self.labels
        result_df['is_exemplar'] = False
        result_df.loc[self.exemplars, 'is_exemplar'] = True
        
        result_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"💾 결과 저장: {output_path}")
        
        return result_df


# ==================== 사용 예시 ====================

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("🎯 개선된 AP 클러스터링 분석")
    print("="*80 + "\n")
    
    # 1. 분석기 초기화
    analyzer = ImprovedAPAnalyzer()
    
    # 2. 데이터 로드 (기존 전처리 함수 사용)
    document_df = analyzer.load_data()
    
    # 3. 방법 1: 여러 유사도 방법 비교 (추천)
    comparison_df = analyzer.compare_similarity_methods(
        methods=['cosine', 'euclidean', 'manhattan', 'rbf'],
        target_clusters=20  # 원하는 군집 수
    )
    
    # 4. 방법 2: 특정 방법으로 직접 실행
    # labels = analyzer.fit_affinity_propagation(
    #     preference=None,  # 자동 탐색
    #     damping=0.9,
    #     method='cosine',  # 또는 'euclidean', 'manhattan', 'rbf'
    #     target_clusters=20
    # )
    
    # 5. 시각화
    analyzer.visualize_clusters(method='tsne', save_path='ap_clusters_improved.png')
    
    # 6. 군집 요약 (상위 5개 군집)
    print("\n" + "="*80)
    print("📋 주요 군집 요약")
    print("="*80 + "\n")
    
    cluster_sizes = Counter(analyzer.labels)
    top_clusters = [c for c, _ in cluster_sizes.most_common(5)]
    
    for cluster_id in top_clusters:
        summary = analyzer.get_cluster_summary(cluster_id)
        print(f"[군집 {cluster_id}] - {summary['n_documents']}개 문서")
        print(f"  상위 키워드: {[kw[0] for kw in summary['top_keywords'][:5]]}")
        print()
    
    # 7. 결과 저장
    result_df = analyzer.export_results('improved_ap_results.csv')
    
    print("\n✅ 모든 분석 완료!")